# Alignment Faking — Kaggle Runner
GPU: T4×2 (32 GB total, tensor-parallel) or P100 (16 GB single).
Set `TENSOR_PARALLEL = 1` if on P100.

In [ ]:
https://github.com/Yeshey/aligment-faking

In [ ]:
# ── Imports + config ──────────────────────────────────────────────────────────
import os, subprocess, sys, time, requests

REPO_URL  = "https://github.com/Yeshey/aligment-faking"
REPO_DIR  = "/kaggle/working/alignment-faking"
VLLM_PORT = 8000

def run(cmd, **kwargs):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=True, **kwargs)
    return result

In [ ]:
# ── HF token ──────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF token loaded from Kaggle Secrets")
except Exception as e:
    print(f"Kaggle Secrets unavailable ({e}) — falling back to .hf_token file")
    if os.path.exists(".hf_token"):
        with open(".hf_token") as f:
            for line in f:
                k, _, v = line.strip().partition("=")
                os.environ[k.strip()] = v.strip()

In [ ]:
# ── Install uv ────────────────────────────────────────────────────────────────
run("curl -LsSf https://astral.sh/uv/install.sh | sh")

os.environ["PATH"] = (
    os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
)

run("uv --version")

In [ ]:
# ── Clone + sync ──────────────────────────────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)

run("uv sync --frozen")

venv_bin = os.path.join(REPO_DIR, ".venv", "bin")
os.environ["PATH"] = venv_bin + ":" + os.environ["PATH"]

sys.path.insert(
    0,
    os.path.join(
        REPO_DIR,
        ".venv",
        "lib",
        f"python{sys.version_info.major}.{sys.version_info.minor}",
        "site-packages",
    ),
)

print("Env ready")

In [ ]:
# ── Load config.env ───────────────────────────────────────────────────────────
from dotenv import load_dotenv

load_dotenv("config.env")

MODEL = os.environ["MODEL"]

In [ ]:
# ── Detect GPU ────────────────────────────────────────────────────────────────
try:
    cc_raw = subprocess.check_output(
        "nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
        shell=True,
    ).decode().strip().split("\n")

    gpu_count = len(cc_raw)
    cc = int(cc_raw[0].replace(".", ""))

except Exception:
    cc = 0
    gpu_count = 1

dtype         = "bfloat16" if cc >= 80 else "float16"
tensor_par    = min(gpu_count, 2)
gpu_mem_util  = 0.90
max_model_len = 8192 if cc >= 80 else 4096

# Enable if OOM
# quant_args = ""
quant_args = "--quantization bitsandbytes --load-format bitsandbytes"

print(
    f"GPU count={gpu_count}, "
    f"compute_cap={cc}, "
    f"dtype={dtype}, "
    f"tensor_par={tensor_par}"
)

In [ ]:
# ── Start vllm ────────────────────────────────────────────────────────────────
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("evaluated_data", exist_ok=True)

chat_template = os.path.join(
    REPO_DIR,
    "chatTemplates",
    "llama3-chat-template.jinja",
)

vllm_cmd = (
    f"vllm serve {MODEL}"
    f" --dtype {dtype}"
    f" --gpu-memory-utilization {gpu_mem_util}"
    f" --max-model-len {max_model_len}"
    f" --tensor-parallel-size {tensor_par}"
    f" --port {VLLM_PORT}"
    f" {quant_args}"
)

print(f"Launching: {vllm_cmd}")

log_out = open("logs/vllm.log", "w")

vllm_proc = subprocess.Popen(
    vllm_cmd,
    shell=True,
    stdout=log_out,
    stderr=log_out,
)

health_url = f"http://localhost:{VLLM_PORT}/health"

for attempt in range(120):
    try:
        if requests.get(health_url, timeout=2).ok:
            print(f"vllm ready (attempt {attempt+1})")
            break

    except Exception:
        pass

    if vllm_proc.poll() is not None:
        log_out.flush()
        raise RuntimeError("vllm died — check logs/vllm.log")

    time.sleep(5)

else:
    raise RuntimeError("vllm not ready after 10 min")

In [ ]:
# ── Run pipeline ──────────────────────────────────────────────────────────────
try:
    os.environ["DRY_RUN"] = "1"
    os.environ["MODEL"] = MODEL
    run("python src/gen.py")
    run("python src/eval.py")
    os.environ.pop("DRY_RUN")

finally:
    vllm_proc.terminate()
    log_out.close()
    print("vllm stopped")